In [8]:
from pyspark.sql import SparkSession

In [9]:
spark = SparkSession.builder.getOrCreate()

In [10]:
data = spark.read.csv(
    "E:/Cluster Computing/Dataset/ca3_logistic_regression_sample.csv",
    header = True,
    inferSchema = True
)

In [11]:
data.show()

+---------+----------+----------+-------------+------+
|StudentID|StudyHours|Attendance|PracticeTests|Result|
+---------+----------+----------+-------------+------+
|      201|         1|        55|            1|  Fail|
|      202|         2|        60|            2|  Fail|
|      203|         2|        68|            1|  Fail|
|      204|         3|        65|            3|  Fail|
|      205|         3|        72|            2|  Fail|
|      206|         4|        75|            3|  Pass|
|      207|         4|        80|            4|  Pass|
|      208|         5|        78|            3|  Pass|
|      209|         5|        85|            4|  Pass|
|      210|         6|        88|            5|  Pass|
|      211|         7|        92|            5|  Pass|
|      212|         8|        95|            6|  Pass|
+---------+----------+----------+-------------+------+



In [12]:
data.printSchema()

root
 |-- StudentID: integer (nullable = true)
 |-- StudyHours: integer (nullable = true)
 |-- Attendance: integer (nullable = true)
 |-- PracticeTests: integer (nullable = true)
 |-- Result: string (nullable = true)



In [13]:
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(
    inputCols = ["StudyHours", "Attendance", "PracticeTests"],
    outputCol = "feature"
)

data = assembler.transform(data)

In [15]:
from pyspark.ml.feature import StandardScaler
scaler = StandardScaler(
    inputCol = "feature",
    outputCol = "scaledFeature"
)

data3 = scaler.fit(data)
data = data3.transform(data)

In [16]:
data.show()

+---------+----------+----------+-------------+------+--------------+--------------------+
|StudentID|StudyHours|Attendance|PracticeTests|Result|       feature|       scaledFeature|
+---------+----------+----------+-------------+------+--------------+--------------------+
|      201|         1|        55|            1|  Fail|[1.0,55.0,1.0]|[0.47061290894652...|
|      202|         2|        60|            2|  Fail|[2.0,60.0,2.0]|[0.94122581789305...|
|      203|         2|        68|            1|  Fail|[2.0,68.0,1.0]|[0.94122581789305...|
|      204|         3|        65|            3|  Fail|[3.0,65.0,3.0]|[1.41183872683958...|
|      205|         3|        72|            2|  Fail|[3.0,72.0,2.0]|[1.41183872683958...|
|      206|         4|        75|            3|  Pass|[4.0,75.0,3.0]|[1.88245163578611...|
|      207|         4|        80|            4|  Pass|[4.0,80.0,4.0]|[1.88245163578611...|
|      208|         5|        78|            3|  Pass|[5.0,78.0,3.0]|[2.35306454473264...|

In [23]:
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer(
    inputCol = "Result",
    outputCol = "ResultIndexed"
)
indexer_data = indexer.fit(data)
data = indexer_data.transform(data)

In [24]:
data.show()

+---------+----------+----------+-------------+------+--------------+--------------------+-------------+
|StudentID|StudyHours|Attendance|PracticeTests|Result|       feature|       scaledFeature|ResultIndexed|
+---------+----------+----------+-------------+------+--------------+--------------------+-------------+
|      201|         1|        55|            1|  Fail|[1.0,55.0,1.0]|[0.47061290894652...|          1.0|
|      202|         2|        60|            2|  Fail|[2.0,60.0,2.0]|[0.94122581789305...|          1.0|
|      203|         2|        68|            1|  Fail|[2.0,68.0,1.0]|[0.94122581789305...|          1.0|
|      204|         3|        65|            3|  Fail|[3.0,65.0,3.0]|[1.41183872683958...|          1.0|
|      205|         3|        72|            2|  Fail|[3.0,72.0,2.0]|[1.41183872683958...|          1.0|
|      206|         4|        75|            3|  Pass|[4.0,75.0,3.0]|[1.88245163578611...|          0.0|
|      207|         4|        80|            4|  Pass|[

In [27]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed = 42)

In [28]:
from pyspark.ml.classification import LogisticRegression
regression = LogisticRegression(
    featuresCol = "scaledFeature",
    labelCol = "ResultIndexed"
    
)

In [29]:
model = regression.fit(train_data)

In [31]:
predictions = model.transform(test_data)

In [32]:
predictions.show()

+---------+----------+----------+-------------+------+--------------+--------------------+-------------+--------------------+--------------------+----------+
|StudentID|StudyHours|Attendance|PracticeTests|Result|       feature|       scaledFeature|ResultIndexed|       rawPrediction|         probability|prediction|
+---------+----------+----------+-------------+------+--------------+--------------------+-------------+--------------------+--------------------+----------+
|      203|         2|        68|            1|  Fail|[2.0,68.0,1.0]|[0.94122581789305...|          1.0|[-51.258928361761...|[5.47684265517184...|       1.0|
|      207|         4|        80|            4|  Pass|[4.0,80.0,4.0]|[1.88245163578611...|          0.0|[38.1761470807992...|           [1.0,0.0]|       0.0|
|      209|         5|        85|            4|  Pass|[5.0,85.0,4.0]|[2.35306454473264...|          0.0|[61.9765119367718...|           [1.0,0.0]|       0.0|
+---------+----------+----------+-------------+-----

In [36]:
from pyspark.ml.evaluation import RegressionEvaluator
metric = ["rmse", "r2", "mse", "mae"]
for i in metric:
    evaluator = RegressionEvaluator(
        labelCol = "ResultIndexed",
        predictionCol = "prediction",
        metricName = i
    )
    print(f"{i} : {evaluator.evaluate(predictions)}")

rmse : 0.0
r2 : 1.0
mse : 0.0
mae : 0.0
